# Vanilla RAG at 20,000 characters - standalone

Re-runs the vanilla RAG baseline with the prompt context raised from **5,000 to 20,000
characters**, matching the budget the 5-channel fusion system gets (`document_head` = 20,000).

### Why this notebook needs nothing from the DGX

Vanilla RAG is **zero-shot**. It touches exactly three things per document: `text`, `country`
and `doc_id`. No interventions, no gold labels, no cross-validation folds, no `Dataset.load()`.

So the entire gold-rebuild chain (`gold.py`, `master.py`) is irrelevant here, and with it the risk
of the intervention set drifting and invalidating the comparison. Everything the run needs is in
two files:

| File | What it is |
|---|---|
| `corpus_documents.json.gz` | all 221 documents, text extracted with pypdf, 12 scanned ones OCR'd |
| `eric_strategies.json` | the 73 ERIC strategies and their 9 categories |

**Scoring is deliberately deferred.** It needs the gold interventions, costs seconds of CPU, and
can be done any time afterwards. The expensive GPU work produces per-document rows that are
independent of the gold, so nothing is lost by separating them.

---
### What changes vs the original 9.2 run

| | Original | Here |
|---|---|---|
| `max_context_chars` | 5,000 | **20,000** |
| `top_k` | 20 | 20 (unchanged) |
| chunk size / overlap | 1,000 / 200 | unchanged |
| encoder | multilingual-e5-large | unchanged |
| generator | Qwen2.5-7B-Instruct | unchanged (4-bit for T4) |
| prompt text | - | unchanged, verbatim |

Two honest differences to record in the write-up: the model is **4-bit quantised** to fit a 16 GB
T4, and the 12 OCR'd documents were re-OCR'd locally with Tesseract 5.4.0, so their text is not
byte-identical to the original run's. That is 12 of 221 documents.


## 1 - Install and check the GPU


In [1]:
!pip -q install 'transformers>=4.44' 'accelerate>=0.33' 'bitsandbytes>=0.43' \
                'sentence-transformers>=3.0' pandas numpy

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('gpu  ', p.name, f'{p.total_memory/1e9:.1f} GB', f'sm_{p.major}{p.minor}')
    if p.major < 8:
        print()
        print('No bf16 hardware on this GPU (T4 is sm_75). Qwen2.5 overflows in fp16 and')
        print('emits "!!!!!" - so we load 4-bit NF4 and smoke-test before the full run.')
else:
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 56.1 MB/s eta 0:00:00
torch 2.11.0+cu128 | cuda True
gpu   NVIDIA L4 23.7 GB sm_89


## 2 - Load the corpus

Upload `corpus_documents.json.gz` and `eric_strategies.json`, or mount Drive and point at them.


In [2]:
from pathlib import Path
import gzip, json

# from google.colab import drive; drive.mount('/content/drive')
BASE = Path('/content')                     # <-- EDIT if the files live elsewhere

CORPUS = BASE / 'corpus_documents.json.gz'
ERIC   = BASE / 'eric_strategies.json'
for f in (CORPUS, ERIC):
    if not f.exists():
        raise SystemExit(f'missing: {f}')

docs = json.loads(gzip.decompress(CORPUS.read_bytes()).decode('utf8'))
eric = json.loads(ERIC.read_text(encoding='utf8'))
print(f'{len(docs)} documents, {sum(d["n_chars"] for d in docs.values()):,} characters')
print(f'{eric["counts"]["strategies"]} strategies in {eric["counts"]["categories"]} categories')


221 documents, 19,796,545 characters
73 strategies in 9 categories


### Verify the corpus


In [3]:
import pandas as pd

rows = []
for c in sorted({d['country'] for d in docs.values()}):
    cd = [d for d in docs.values() if d['country'] == c]
    rows.append({'country': c, 'documents': len(cd),
                 'OCR': sum(1 for d in cd if d['source'] == 'ocr'),
                 'chars': sum(d['n_chars'] for d in cd),
                 'median_chars': int(pd.Series([d['n_chars'] for d in cd]).median())})
tab = pd.DataFrame(rows).sort_values('documents', ascending=False)
tab.loc['TOTAL'] = ['TOTAL', tab.documents.sum(), tab.OCR.sum(), tab.chars.sum(), '']
display(tab)
print('EXPECTED: 221 documents across 13 countries, 12 OCR-recovered')


,country,documents,OCR,chars,median_chars
0,Bangladesh,23,3,2987303,72412
1,Brazil,23,0,2883023,35080
2,China,23,0,275128,3040
7,Pakistan,20,2,1425568,42098
4,India,18,1,1379955,47193
6,Malawi,18,0,2534673,47754
3,Ghana,17,0,1078813,21755
12,Zambia,15,0,556505,4414
9,Russia,15,1,2325458,52578
8,Philippines,14,1,1283839,44579


EXPECTED: 221 documents across 13 countries, 12 OCR-recovered


## 3 - Configuration

`top_k=20` at 1,000 chars per chunk yields exactly 20,000 characters, so retrieval depth and the
context cap now agree. In the original run `top_k` was already 20, but the 5,000-char cap threw
away three quarters of everything that had been retrieved.


In [4]:
MAX_CONTEXT_CHARS = 20_000      # was 5_000
TOP_K             = 20
CHUNK_SIZE        = 1_000
CHUNK_OVERLAP     = 200
MAX_CHUNKS        = 400
ENCODER           = 'intfloat/multilingual-e5-large'
MODEL             = 'Qwen/Qwen2.5-7B-Instruct'
MAX_NEW_TOKENS    = 600
BATCH_SIZE        = 4           # raise to 4 if VRAM allows

# one fixed query for the whole task, verbatim from vanilla_rag.QUERY
QUERY = ('antimicrobial resistance intervention: its name, objectives, activities, '
         'implementation and monitoring')

OUT = BASE / 'vanilla_rag_20k'
OUT.mkdir(parents=True, exist_ok=True)
print('per-document rows ->', OUT)
print('already written   :', len(list(OUT.glob('*.json'))), 'of', len(docs))


per-document rows -> /content/vanilla_rag_20k
already written   : 0 of 221


## 4 - The pipeline, verbatim from `vanilla_rag.py`

`chunk()` is from `sota_rag.py:43`; `prompt()` and the JSON parsing are from `vanilla_rag.py`.
Reproduced inline so the notebook has no repo dependency - the text is unchanged.


In [5]:
import re, numpy as np

def chunk(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i + size])
        i += size - overlap
    return out


STRATEGIES = [s['strategy'] for s in eric['strategies']]
S2C = {s['strategy']: s['category'] for s in eric['strategies']}

# eric.prompt_block_flat()
TAXONOMY = '\n'.join(f'{i}. {s}  ({S2C[s]})' for i, s in enumerate(STRATEGIES, 1))


import unicodedata

def norm(s):
    """eric.norm - verbatim."""
    s = unicodedata.normalize('NFKD', str(s))
    s = s.replace('\u2019', "'").replace('\u2018', "'")
    return re.sub(r'[^a-z0-9]', '', s.lower())


_BY_NORM = {norm(s): s for s in STRATEGIES}


def canonical(label):
    """eric.canonical - verbatim. The fuzzy fallback matters: a 7B model routinely
    emits trailing punctuation, doubled spaces or a stray leading article, and an
    exact-match parser would silently discard those as non-codes - depressing the
    score for a reason that has nothing to do with the model."""
    if not label or not str(label).strip():
        return None
    key = norm(label)
    hit = _BY_NORM.get(key)
    if hit:
        return hit
    # tolerate a missing/extra leading article or trailing punctuation
    for cand_norm, cand in _BY_NORM.items():
        if key and (key in cand_norm or cand_norm in key) and abs(len(key) - len(cand_norm)) <= 4:
            return cand
    return None


def build_prompt(doc, context):
    ctx = '\n---\n'.join(context)[:MAX_CONTEXT_CHARS]
    return (
        f'You are a policy analyst coding a national health policy document from '
        f'{doc["country"]} into a structured database.\n\n'
        f'THE 73 ERIC STRATEGIES (choose only from this numbered list):\n{TAXONOMY}\n\n'
        f'RETRIEVED PASSAGES FROM THE DOCUMENT:\n{ctx}\n\n'
        'Record the main intervention this document establishes.\n\n'
        'Field instructions:\n'
        '- name: the official title of the programme, in English\n'
        '- summary: 2-3 sentences on objectives, who runs it, and main activities\n'
        '- focus: exactly ONE of: AMR, AMU, IPC, Antimicrobial Stewardship, '
        'Public Education & Awareness Campaign\n'
        '- level: exactly ONE of: Macro, Meso, Micro\n'
        '- maturity: exactly ONE of: Developed, Implemented, Evaluated\n'
        '- eric_codes: a list of 8 to 12 strategy names copied EXACTLY from the '
        'numbered list above. Use ONLY the strategy names, never the category '
        'names shown in brackets. Never write placeholder text.\n\n'
        'Reply with a single JSON object using those six keys and real values.\nJSON:')


def parse_reply(reply):
    m = re.search(r'\{.*\}', reply, re.S)
    rec = {'name': '', 'summary': '', 'eric_codes': []}
    if m:
        try:
            obj = json.loads(m.group(0))
            codes = [canonical(c) for c in obj.get('eric_codes', [])]
            rec = {'name': str(obj.get('name', '')).strip(),
                   'summary': str(obj.get('summary', '')).strip(),
                   'focus': obj.get('focus'), 'level': obj.get('level'),
                   'maturity': obj.get('maturity'),
                   'eric_codes': [c for c in codes if c]}
        except json.JSONDecodeError:
            pass
    return rec

print('taxonomy block:', len(TAXONOMY), 'chars |', len(STRATEGIES), 'strategies')


taxonomy block: 5100 chars | 73 strategies


## 5 - Retrieval

Chunk embeddings are cached to disk, so a disconnect never costs the same embedding twice, and
you can re-run generation without redoing retrieval.


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

enc = SentenceTransformer(ENCODER, device='cuda')
enc.max_seq_length = 256

QVEC = np.asarray(enc.encode(['query: ' + QUERY], normalize_embeddings=True,
                             show_progress_bar=False))[0]

CACHE = BASE / 'chunk_cache'
CACHE.mkdir(exist_ok=True)


def retrieve(doc_id, doc):
    chunks = chunk(doc['text'])[:MAX_CHUNKS]
    if len(chunks) <= TOP_K:
        return chunks
    f = CACHE / f'{doc_id}.npy'
    if f.exists():
        V = np.load(f)
    else:
        V = np.asarray(enc.encode(['passage: ' + c for c in chunks],
                                  normalize_embeddings=True, batch_size=64,
                                  show_progress_bar=False))
        np.save(f, V)
    idx = np.argsort(-(V @ QVEC))[:TOP_K]
    return [chunks[i] for i in sorted(idx)]      # restored to document order


probe_id = sorted(docs)[0]
ctx = retrieve(probe_id, docs[probe_id])
print(f'{probe_id}: {len(ctx)} chunks -> prompt context '
      f'{len(chr(10).join(ctx)[:MAX_CONTEXT_CHARS]):,} chars')


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Bangladesh_01: 20 chunks -> prompt context 19,303 chars


In [7]:
# Precompute every context while the encoder is the only model on the GPU,
# then free it. Two models resident at once is what caused the T4 OOM.
import gc, json, time

CTX = BASE / 'contexts.json'
if CTX.exists():
    contexts = json.loads(CTX.read_text())
else:
    contexts, t0 = {}, time.time()
    for i, d in enumerate(sorted(docs), 1):
        contexts[d] = retrieve(d, docs[d])
        if i % 25 == 0 or i == len(docs):
            print(f'  retrieved {i}/{len(docs)}  {time.time()-t0:.0f}s', flush=True)
    CTX.write_text(json.dumps(contexts))
print('contexts ready:', len(contexts))

del enc
gc.collect(); torch.cuda.empty_cache()
print(f'encoder freed -> {torch.cuda.memory_allocated()/1e9:.1f} GB')

def retrieve(doc_id, doc):
    return contexts[doc_id]

  retrieved 25/221  63s
  retrieved 50/221  105s
  retrieved 75/221  122s
  retrieved 100/221  166s
  retrieved 125/221  219s
  retrieved 150/221  276s
  retrieved 175/221  335s
  retrieved 200/221  397s
  retrieved 221/221  424s
contexts ready: 221
encoder freed -> 0.0 GB


## 6 - Load the generator, then smoke-test

**Do not skip this.** Three documents takes a couple of minutes and tells you whether an overnight
run is worth starting. If the output is exclamation marks, that is the fp16 overflow and the run
is worthless.


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, time

tok = AutoTokenizer.from_pretrained(MODEL, padding_side='left')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# L4 is Ada (sm_89) with native bf16 - no quantisation, exactly as the original ran.
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    device_map='cuda',
    attn_implementation='sdpa')
model.eval()
print(f'loaded bf16: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')


_truncated = []

def _decode_batch(batch, max_len):
    texts = [tok.apply_chat_template([{'role': 'user', 'content': p}],
                                     tokenize=False, add_generation_prompt=True)
             for p in batch]
    e = tok(texts, return_tensors='pt', padding=True, truncation=True,
            max_length=max_len).to(model.device)
    with torch.no_grad():
        g = model.generate(**e, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           temperature=None, top_p=None, pad_token_id=tok.pad_token_id)
    s = e['input_ids'].shape[1]
    return [tok.decode(x[s:], skip_special_tokens=True) for x in g]


def generate_many(prompts, batch_size=BATCH_SIZE):
    """Batched, with an OOM ladder that tries FULL length first so context is
    never silently truncated just because a batch was too large."""
    out, old = [], tok.padding_side
    tok.padding_side = 'left'
    try:
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            try:
                out += _decode_batch(chunk, 8192)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                for j, one in enumerate(chunk):
                    for max_len in (8192, 6144, 4096, 2048):
                        try:
                            out += _decode_batch([one], max_len)
                            if max_len < 8192:
                                _truncated.append((i + j, max_len))
                                print(f'    doc {i+j} truncated to {max_len}', flush=True)
                            break
                        except torch.cuda.OutOfMemoryError:
                            torch.cuda.empty_cache()
                    else:
                        _truncated.append((i + j, 0)); out.append('')
                        print(f'    doc {i+j} unrecoverable OOM', flush=True)
            torch.cuda.empty_cache()
    finally:
        tok.padding_side = old
    return out

print('ready')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

loaded bf16: 15.2 GB allocated
ready


In [9]:
probe = sorted(docs)[:3]
t0 = time.time()
prompts = [build_prompt(docs[d], retrieve(d, docs[d])) for d in probe]
print('prompt tokens:', [len(tok(p)['input_ids']) for p in prompts])
replies = generate_many(prompts, batch_size=1)
dt = time.time() - t0

recs = {d: parse_reply(r) for d, r in zip(probe, replies)}
for d, r in recs.items():
    print()
    print('---', d, f"({docs[d]['country']})")
    print('  name :', (r.get('name') or '')[:88])
    print('  codes:', len(r.get('eric_codes', [])), r.get('eric_codes', [])[:3])

bang  = sum(1 for r in recs.values() if '!!!' in (r.get('name') or ''))
blank = sum(1 for r in recs.values() if not r.get('eric_codes'))
print()
print(f'{dt:.0f}s for 3 docs -> about {dt/3*len(docs)/3600:.1f} h for all {len(docs)}')
if bang:
    print('FP16 OVERFLOW - do not proceed.')
elif blank == 3:
    print('No codes parsed from any of the three - check the reply format before proceeding.')
else:
    print('Healthy. Proceed to section 7.')


prompt tokens: [5359, 2383, 5693]

--- Bangladesh_01 (Bangladesh)
  name : National Drug Policy 2016
  codes: 8 ['Use data experts', 'Use data warehousing techniques', 'Develop academic partnerships']

--- Bangladesh_02 (Bangladesh)
  name : Bangladesh Pharmacy Model Initiative
  codes: 0 []

--- Bangladesh_03 (Bangladesh)
  name : National Action Plan for Antimicrobial Resistance Containment in Bangladesh
  codes: 2 ['Promote adaptability', 'Use data experts']

35s for 3 docs -> about 0.7 h for all 221
Healthy. Proceed to section 7.


In [10]:
d = 'Bangladesh_02'
p = build_prompt(docs[d], retrieve(d, docs[d]))
raw = generate_many([p], batch_size=1)[0]
print('--- RAW MODEL REPLY ---')
print(raw[:1500])
print('\n--- PARSED ---')
print(parse_reply(raw))

import re as _re
m = _re.search(r'\{.*\}', raw, _re.S)
if m:
    try:
        obj = json.loads(m.group(0))
        codes = obj.get('eric_codes', [])
        print(f'\nmodel emitted {len(codes)} codes:')
        for c in codes:
            print(f'   {c!r:55s} -> {canonical(c)!r}')
    except json.JSONDecodeError as e:
        print('\nJSON PARSE FAILED:', e)

--- RAW MODEL REPLY ---
```json
{
  "name": "Bangladesh Pharmacy Model Initiative",
  "summary": "The Bangladesh Pharmacy Model Initiative aims to standardize and accredit private retail drug outlets, enhancing regulatory oversight and consumer access to quality pharmaceutical services. It is run by the Ministry of Health and Family Welfare with technical support from Management Sciences for Health and funded by DFID through JDTAF.",
  "focus": "Public Education & Awareness Campaign",
  "level": "Macro",
  "maturity": "Implemented",
  "eric_codes": ["Change infrastructure", "Develop stakeholder interrelationships", "Engage consumers", "Provide interactive assistance", "Train and educate stakeholders", "Use evaluative and iterative strategies", "Utilize financial strategies"]
}
```

--- PARSED ---
{'name': 'Bangladesh Pharmacy Model Initiative', 'summary': 'The Bangladesh Pharmacy Model Initiative aims to standardize and accredit private retail drug outlets, enhancing regulatory oversig

## 7 - Main run

Resumable. Documents already written to `OUT` are skipped, so a Colab disconnect costs only the
batch in flight. Re-run this cell to continue.


In [11]:
todo = [d for d in sorted(docs) if not (OUT / f'{d}.json').exists()]
print(f'{len(todo)} to do, {len(docs) - len(todo)} already written')

t0 = time.time()
for i in range(0, len(todo), 10):
    batch = todo[i:i + 10]
    prompts = [build_prompt(docs[d], retrieve(d, docs[d])) for d in batch]
    replies = generate_many(prompts, batch_size=BATCH_SIZE)
    for d, r in zip(batch, replies):
        rec = parse_reply(r)
        rec['_raw'] = r[:4000]          # keep the raw reply for diagnosis
        (OUT / f'{d}.json').write_text(json.dumps(rec, ensure_ascii=False))
    done = min(i + 10, len(todo))
    el = time.time() - t0
    print(f'  {done}/{len(todo)}  {el/60:.1f} min elapsed  '
          f'~{el/done*(len(todo)-done)/60:.0f} min left', flush=True)
print('done')

221 to do, 0 already written
  10/221  1.8 min elapsed  ~39 min left
  20/221  3.7 min elapsed  ~37 min left
  30/221  6.7 min elapsed  ~43 min left
  40/221  9.7 min elapsed  ~44 min left
  50/221  13.1 min elapsed  ~45 min left
  60/221  16.3 min elapsed  ~44 min left
  70/221  19.2 min elapsed  ~41 min left
  80/221  21.2 min elapsed  ~37 min left
  90/221  22.9 min elapsed  ~33 min left
  100/221  26.6 min elapsed  ~32 min left
  110/221  29.9 min elapsed  ~30 min left
  120/221  32.6 min elapsed  ~27 min left
  130/221  34.4 min elapsed  ~24 min left
  140/221  36.1 min elapsed  ~21 min left
  150/221  38.1 min elapsed  ~18 min left
  160/221  39.9 min elapsed  ~15 min left
  170/221  42.6 min elapsed  ~13 min left
  180/221  47.0 min elapsed  ~11 min left
  190/221  49.6 min elapsed  ~8 min left
  200/221  51.6 min elapsed  ~5 min left
  210/221  53.5 min elapsed  ~3 min left
  220/221  54.9 min elapsed  ~0 min left
  221/221  55.1 min elapsed  ~0 min left
done


## 8 - Package the results

Download `vanilla_rag_20k_rows.json`. Scoring happens separately, against the gold interventions:
extraction F1, classification, and ERIC F1 both **end-to-end** and **isolated** (see the notes on
why RAG's 9.2 is not directly comparable to the ladder's 54.0).


In [12]:
rows = {f.stem: json.loads(f.read_text()) for f in OUT.glob('*.json')}

blob = {'run': 'vanilla_rag_20k',
        'config': {'max_context_chars': MAX_CONTEXT_CHARS, 'top_k': TOP_K,
                   'chunk': CHUNK_SIZE, 'overlap': CHUNK_OVERLAP,
                   'max_chunks': MAX_CHUNKS, 'encoder': ENCODER,
                   'model': MODEL, 'quantisation': '4bit-nf4',
                   'max_new_tokens': MAX_NEW_TOKENS},
        'n_documents': len(rows),
        'rows': rows}

out_file = BASE / 'vanilla_rag_20k_rows.json'
out_file.write_text(json.dumps(blob, indent=1, ensure_ascii=False))
print('wrote', out_file, f'({out_file.stat().st_size/1e6:.1f} MB)')

n_codes = [len(r.get('eric_codes', [])) for r in rows.values()]
named   = sum(1 for r in rows.values() if r.get('name'))
empty   = sum(1 for n in n_codes if n == 0)
print()
print(f'documents with a name : {named}/{len(rows)}')
print(f'documents with 0 codes: {empty}/{len(rows)}')
if n_codes:
    print(f'codes per document    : mean {sum(n_codes)/len(n_codes):.1f}, '
          f'min {min(n_codes)}, max {max(n_codes)}')
print()
print('Gold averages 8.9 codes per intervention. The prompt asks for 8 to 12.')

# from google.colab import files; files.download(str(out_file))


wrote /content/vanilla_rag_20k_rows.json (0.4 MB)

documents with a name : 197/221
documents with 0 codes: 28/221
codes per document    : mean 6.4, min 0, max 12

Gold averages 8.9 codes per intervention. The prompt asks for 8 to 12.


In [15]:
import shutil

In [17]:
import shutil
from google.colab import files

shutil.make_archive('/content/chunk_cache', 'zip', '/content/chunk_cache')
files.download('/content/chunk_cache.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import shutil
from google.colab import files

shutil.make_archive('/content/vanilla_rag_20k', 'zip', '/content/vanilla_rag_20k')
print('size:', round(__import__('os').path.getsize('/content/vanilla_rag_20k.zip')/1e6, 2), 'MB')
files.download('/content/vanilla_rag_20k.zip')

size: 0.14 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>